### HuggingFace Transformers

In [1]:
!pip install "transformers"
#Checl CUDA version with nvidia-smi
!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu130 #for CUDA version 13.0
#!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu126 #for CUDA version 12.6
!pip install --upgrade pip

Looking in indexes: https://artifacts.dell.com/artifactory/api/pypi/python/simple, https://artifacts.dell.com/artifactory/api/pypi/ailfc-1003745-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aia-1001238-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aiops-1002685-pypi-prd-local/simple
Looking in indexes: https://download.pytorch.org/whl/cu130, https://artifacts.dell.com/artifactory/api/pypi/ailfc-1003745-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aia-1001238-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aiops-1002685-pypi-prd-local/simple
Looking in indexes: https://artifacts.dell.com/artifactory/api/pypi/python/simple, https://artifacts.dell.com/artifactory/api/pypi/ailfc-1003745-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aia-1001238-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aiops-1002685-pypi-prd-local/simple


In [2]:
import torch
torch.set_num_threads(12)

USE_CUDA = torch.cuda.is_available()
if USE_CUDA:
    print(f"CUDA Available: {USE_CUDA}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    
    current_device = torch.cuda.current_device()
    
    print(f"Current Device ID: {current_device}")
    print(f"Device Name: {torch.cuda.get_device_name(current_device)}")
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    #torch.backends.cuda.enable_math_sdp(True)
else:
    print("PyTorch is falling back to the CPU.")

from transformers import pipeline

CUDA Available: True
GPU Count: 1
Current Device ID: 0
Device Name: NVIDIA H100 80GB HBM3 MIG 1g.10gb


In [3]:
sentiment_classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [4]:
sentiment_classifier("I'm so excited to be learning about large language models")

[{'label': 'POSITIVE', 'score': 0.9997096657752991}]

In [5]:
ner = pipeline("ner", model = "dslim/bert-base-NER")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
zeroshot_classifier = pipeline("zero-shot-classification", model = "facebook/bart-large-mnli")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [7]:
sequence_to_classify = "one day I will see the world"
candidate_labels = ['travel', 'cooking', 'dancing']
zeroshot_classifier(sequence_to_classify, candidate_labels)

{'sequence': 'one day I will see the world',
 'labels': ['travel', 'dancing', 'cooking'],
 'scores': [0.9938651323318481, 0.0032737816218286753, 0.002861041808500886]}

### Pre-trained Tokenizers

In [8]:
from transformers import AutoTokenizer
model = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model)

In [9]:
sentence = "I'm so excited to be learning about large language models"
input_ids = tokenizer(sentence)
print(input_ids)

{'input_ids': [101, 1045, 1005, 1049, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653, 4275, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [10]:
tokens = tokenizer.tokenize(sentence)
print(tokens)

['i', "'", 'm', 'so', 'excited', 'to', 'be', 'learning', 'about', 'large', 'language', 'models']


In [11]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

[1045, 1005, 1049, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653, 4275]


In [12]:
decoded_ids = tokenizer.decode(token_ids)
print(decoded_ids)

i ' m so excited to be learning about large language models


In [13]:
tokenizer.decode(101)

'[CLS]'

In [14]:
tokenizer.decode(102)

'[SEP]'

In [15]:
model_2 = "xlnet-base-cased"
tokenizer2 = AutoTokenizer.from_pretrained(model_2)

In [16]:
input_ids = tokenizer2(sentence)
print(input_ids)

{'input_ids': [35, 26, 98, 102, 5564, 22, 39, 1899, 75, 392, 1243, 2626, 4, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [17]:
tokens = tokenizer2.tokenize(sentence)
print(tokens)

['▁I', "'", 'm', '▁so', '▁excited', '▁to', '▁be', '▁learning', '▁about', '▁large', '▁language', '▁models']


In [18]:
token_ids = tokenizer2.convert_tokens_to_ids(tokens)
print(token_ids)

[35, 26, 98, 102, 5564, 22, 39, 1899, 75, 392, 1243, 2626]


In [19]:
tokenizer2.decode(4)

'<sep>'

In [20]:
tokenizer2.decode(3)

'<cls>'

### Huggingface and Pytorch/Tensorflow

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [22]:
print(sentence)
print(input_ids)

I'm so excited to be learning about large language models
{'input_ids': [35, 26, 98, 102, 5564, 22, 39, 1899, 75, 392, 1243, 2626, 4, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [23]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
input_ids_pt = tokenizer(sentence, return_tensors ="pt")
print(input_ids_pt)

{'input_ids': tensor([[ 101, 1045, 1005, 1049, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653,
         4275,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [24]:
model_3 = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

with torch.no_grad():
    logits = model_3(**input_ids_pt).logits

predicted_class_id = logits.argmax().item()
model_3.config.id2label[predicted_class_id]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

'POSITIVE'

### Saving and loading models

In [25]:
model_directory = "my_saved_models"
tokenizer.save_pretrained(model_directory)

('my_saved_models/tokenizer_config.json', 'my_saved_models/tokenizer.json')

In [27]:
#model.save_pretrained(model_directory)
#model_2.save_pretrained(model_directory)
model_3.save_pretrained(model_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [28]:
my_tokenizer = AutoTokenizer.from_pretrained(model_directory)

In [31]:
my_model = AutoModelForSequenceClassification.from_pretrained(model_directory)
print(my_model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
